# Agentic fighters

## LangGraph for simplicity

### Graphs

![Graphs](./../images/Graphs.png)

Important concepts:

- nodes
- relations
- state
- persistence

In [1]:
import os

In [2]:
if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("Please set OPENAI_API_KEY environment variable")

LLM_MODEL = "gpt-4o-mini"
LLM_TEMPERATURE = 0

In [3]:
from IPython.display import Markdown
from typing import TypedDict, Annotated
import operator

from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph, END

In [4]:
base_model = ChatOpenAI(model=LLM_MODEL, temperature=LLM_TEMPERATURE)

#### State

In [5]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

#### Tools

In [6]:
if not os.environ.get("TAVILY_API_KEY"):
    raise ValueError("Please set OPENAI_API_KEY environment variable")

tool = TavilySearchResults(max_results=4)
print(type(tool))
print(tool.name)

<class 'langchain_community.tools.tavily_search.tool.TavilySearchResults'>
tavily_search_results_json


#### Workflow with the graph

In [7]:
class Agent:
    # General architecture of the graph
    ####################################
    def __init__(self, model, tools, system=""):
        self.system = system
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

        graph = StateGraph(AgentState)

        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)

        graph.add_conditional_edges(
            "llm",
            self.exists_action,
            {True: "action", False: END}
        )
        graph.add_edge("action", "llm")

        graph.set_entry_point("llm")

        self.graph = graph.compile()


    # Main nodes
    #############
    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)

        return {'messages': [message]}

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            if t['name'] not in self.tools:
                print("\n ....bad tool name....")
                result = "bad tool name, retry"
            else:
                result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")

        return {'messages': results}


    # Conditional edges
    ####################
    def exists_action(self, state: AgentState):
        result = state['messages'][-1]

        return len(result.tool_calls) > 0

In [8]:
prompt = """You are an intelligent search assistant. Use the search engine to search for information. \
You are allowed to make multiple calls (either together or sequentially). \
You should only search for information when you are sure of what you need. \
You are allowed to search for information before asking a question for clarification.
"""

abot = Agent(base_model, [tool], system=prompt)

In [21]:
from IPython.display import Image

Image(abot.graph.get_graph().draw_mermaid_png())

ReadTimeout: HTTPSConnectionPool(host='mermaid.ink', port=443): Read timed out. (read timeout=10)

In [11]:
messages = [HumanMessage(content="What's the weather like in Valencia?")]
result = abot.graph.invoke({"messages": messages})

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Valencia'}, 'id': 'call_nNpiROq7vBueAWqjE5RRfECX', 'type': 'tool_call'}
Back to the model!


In [12]:
result

{'messages': [HumanMessage(content="What's the weather like in Valencia?", additional_kwargs={}, response_metadata={}),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_nNpiROq7vBueAWqjE5RRfECX', 'function': {'arguments': '{"query":"current weather in Valencia"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 145, 'total_tokens': 167, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0392822090', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-e6a2a744-5679-461b-ba19-d378c08a442b-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Valencia'}, 'id': 'call_nNpiROq7vBueAWqjE5RRfECX', 'type':

In [13]:
result['messages'][-1].content

'The current weather in Valencia, Spain is as follows:\n\n- **Temperature**: 70°F (21.2°C)\n- **Condition**: Sunny\n- **Feels Like**: 70°F (21.2°C)\n- **Humidity**: 64%\n- **Wind**: 7 mph (11.3 km/h) from the Southeast\n- **Pressure**: 29.89 inHg\n- **Dew Point**: 57°F (14°C)\n\nOverall, it is a pleasant day with clear skies and comfortable temperatures. For more details, you can check the full report [here](https://www.timeanddate.com/weather/spain/valencia).'

In [14]:
messages = [HumanMessage(content="What's the weather in Valencia, Sarajevo, Lausanne and Zurich?")]
result = abot.graph.invoke({"messages": messages})

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Valencia'}, 'id': 'call_s8LuAKkdH2K9EFk9PqQCiud2', 'type': 'tool_call'}
Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Sarajevo'}, 'id': 'call_wiGWu6oH7qiDIwKpQhmY6fpJ', 'type': 'tool_call'}
Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Lausanne'}, 'id': 'call_wx2u2pqngJzunLaIFd7CG9WJ', 'type': 'tool_call'}
Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in Zurich'}, 'id': 'call_5JnajNwyX8qoZt7e1CMYDlxH', 'type': 'tool_call'}
Back to the model!


In [15]:
Markdown(result['messages'][-1].content)

Here is the current weather information for Valencia, Sarajevo, Lausanne, and Zurich:

### Valencia, Spain
- **Temperature**: 21.2°C (70.2°F)
- **Condition**: Sunny
- **Feels Like**: 21.2°C (70.2°F)
- **Wind**: 10.8 km/h (6.7 mph) from the ESE
- **Humidity**: 64%
- **Pressure**: 1012 hPa
- **More Info**: [Time and Date - Valencia Weather](https://www.timeanddate.com/weather/spain/valencia)

### Sarajevo, Bosnia-Herzegovina
- **Temperature**: 16.1°C (61°F)
- **Condition**: Scattered clouds
- **Feels Like**: 17.2°C (63°F)
- **Wind**: 12.9 km/h (8 mph) from the North
- **Humidity**: 31%
- **Pressure**: 1015 hPa
- **More Info**: [Time and Date - Sarajevo Weather](https://www.timeanddate.com/weather/bosnia-herzegovina/sarajevo)

### Lausanne, Switzerland
- **Temperature**: 19.3°C (66.7°F)
- **Condition**: Patchy rain nearby
- **Feels Like**: 19.3°C (66.7°F)
- **Wind**: 3.6 km/h (2.2 mph) from the North
- **Humidity**: 52%
- **Pressure**: 1015 hPa
- **More Info**: [Time and Date - Lausanne Weather](https://www.timeanddate.com/weather/switzerland/lausanne)

### Zurich, Switzerland
- **Temperature**: 18.1°C (64.6°F)
- **Condition**: Partly cloudy
- **Feels Like**: 18.1°C (64.6°F)
- **Wind**: 7.9 km/h (4.9 mph) from the ENE
- **Humidity**: 60%
- **Pressure**: 1015 hPa
- **More Info**: [Time and Date - Zurich Weather](https://www.timeanddate.com/weather/switzerland/zurich)

If you need more detailed forecasts or additional information, feel free to ask!

In [16]:
query = "Who won the super bowl in 2024? In what state is the winning team headquarters located? \
What is the GDP of that state? Answer each question."
messages = [HumanMessage(content=query)]

abot = Agent(base_model, [tool], system=prompt)
result = abot.graph.invoke({"messages": messages})

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'Super Bowl 2024 winner'}, 'id': 'call_bzYiMarN30CkV5hsK8f7Kt9u', 'type': 'tool_call'}
Back to the model!
Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'Kansas City Chiefs headquarters location'}, 'id': 'call_iVAuepG1fhB3U1799A9YNcIn', 'type': 'tool_call'}
Back to the model!
Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'GDP of Missouri 2024'}, 'id': 'call_2cXcZIEdLAUXmnV7om0TZUIT', 'type': 'tool_call'}
Back to the model!


In [17]:
print(result['messages'][-1].content)

1. **Who won the Super Bowl in 2024?**
   - The Kansas City Chiefs won the Super Bowl in 2024, defeating the San Francisco 49ers with a score of 25-22 in overtime.

2. **In what state is the winning team's headquarters located?**
   - The Kansas City Chiefs are headquartered in Missouri.

3. **What is the GDP of that state?**
   - As of late 2024, Missouri's GDP reached approximately $455 billion.
